In [ ]:
import os, platform, sys
assert os.path.exists('/content'), 'Open this notebook in Google Colab.'
assert (3, 10) <= sys.version_info[:2] < (3, 14), 'Use Colab Python 3.10 through 3.13.'
print('Python', platform.python_version(), '| Colab environment ready')


In [ ]:
!rm -rf /content/babel
!git clone --quiet https://github.com/dhelmy990/babel.git /content/babel
!git -C /content/babel checkout --quiet 92f3ac697d78eb827d75b033df92dcbed887def7
!python -m pip install --quiet --require-hashes -r /content/babel/training/requirements-colab.lock
!python -m pip install --quiet --no-deps -e /content/babel/training
REPOSITORY_URL = 'https://github.com/dhelmy990/babel.git'
SOURCE_COMMIT_SHA = '92f3ac697d78eb827d75b033df92dcbed887def7'


In [ ]:
import subprocess
installed_source_sha = subprocess.check_output(
    ['git', '-C', '/content/babel', 'rev-parse', 'HEAD'], text=True
).strip()
assert installed_source_sha == SOURCE_COMMIT_SHA
from babel_training.config import DistillationConfig
print('Pinned Babel source imported:', installed_source_sha)


In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
assert HF_TOKEN, 'Add HF_TOKEN under the key icon in the Colab left sidebar.'
print('Private Hub credential loaded from Colab Secrets (value hidden).')


In [ ]:
from datetime import datetime, timezone
from google.colab import drive
drive.mount('/content/drive')
run_id = datetime.now(timezone.utc).strftime('interview-50k-%Y%m%dT%H%M%SZ')
drive_root = '/content/drive/MyDrive/babel-distillation/interview-50k'
run_root = os.path.join(drive_root, run_id)
os.makedirs(run_root, exist_ok=False)
print('Drive run directory:', run_root)


In [ ]:
DATASET_REPO_ID = 'dhelmy990/babel-wikipedia-experiment'
DATASET_CONFIG = 'distillation_2016_interview'
DATASET_REVISION = 'b440e98b04ab77afed7caf0455eca3189235fc3b'
MODEL_REVISION = '97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3'
MANIFEST_SHA256 = '33c65554da38af5888e5aae75350ae8ee7889d6047c9f8339d97781e4326de09'
TRAIN_ORDERED_SHA256 = '518c30f10859a88681c3708ab0236bd104fdde96acff09089515d871d9600a1e'
VALIDATION_ORDERED_SHA256 = '64cd7c82c58d73947f24b8120ef3c2e5c3a4a8f145bf0a7a6522175bcd1b2cd6'
TEST_ORDERED_SHA256 = 'd2cd61ee895c2f6386c708d7884666b4aa579174674e8bf70e876ee891956bf5'
TRAIN_PARQUET_SHA256 = '11a217879913305a88b0bfaafffa39f132883d2b6f27252a02054ba95ea6b2c5'
VALIDATION_PARQUET_SHA256 = 'a925eb795f253635f3a80a76994a7139a0f81f4e784beb61dfc93f8b662dc8f0'
TEST_PARQUET_SHA256 = '103f22b38b048973f8ab6ba52efca41f667f37c305b5dfea8b752dc492d7ac03'
EXPECTED_COUNTS = {'total': 60_000, 'train': 50_000, 'validation': 5_000, 'test': 5_000}
EXPECTED_ORDERED_SHA256 = {
    'train': TRAIN_ORDERED_SHA256,
    'validation': VALIDATION_ORDERED_SHA256,
    'test': TEST_ORDERED_SHA256,
}
EXPECTED_PARQUET_SHA256 = {
    'train': TRAIN_PARQUET_SHA256,
    'validation': VALIDATION_PARQUET_SHA256,
    'test': TEST_PARQUET_SHA256,
}
SMOKE_ROWS = 1_000
TRAIN_ROWS = 50_000
VALIDATION_ROWS = 5_000
TEST_COUNT = 5_000
EPOCHS = 1


In [ ]:
config = DistillationConfig(max_length=384)
assert config.model_revision == MODEL_REVISION
per_device_batch_size = 2
gradient_accumulation_steps = 8
checkpoint_interval = 100
max_runtime_minutes = None
TRAINING_SEED = 7
DESTINATION_MODEL_REPO = 'dhelmy990/babel-qwen-navigation-2016-interview'
training_config = {
    'config_version': 1,
    'source_commit_sha': SOURCE_COMMIT_SHA,
    'model_id': config.model_id,
    'model_revision': config.model_revision,
    'tokenizer_revision': config.model_revision,
    'dataset_repo_id': DATASET_REPO_ID,
    'dataset_config': DATASET_CONFIG,
    'dataset_commit_sha': DATASET_REVISION,
    'dataset_manifest_sha256': MANIFEST_SHA256,
    'ordered_identity_sha256': EXPECTED_ORDERED_SHA256,
    'parquet_sha256': EXPECTED_PARQUET_SHA256,
    'teacher_dimension': config.teacher_dimension,
    'projection_input_dimension': 1024,
    'projection_output_dimension': 100,
    'max_length': config.max_length,
    'lambda_rel': config.lambda_rel,
    'lora_rank': config.lora_rank,
    'lora_alpha': config.lora_alpha,
    'lora_dropout': config.lora_dropout,
    'lora_targets': list(config.lora_targets),
    'lora_bias': 'none',
    'per_device_batch_size': per_device_batch_size,
    'gradient_accumulation_steps': gradient_accumulation_steps,
    'checkpoint_interval_optimizer_steps': checkpoint_interval,
    'smoke_rows': SMOKE_ROWS,
    'train_rows': TRAIN_ROWS,
    'validation_rows': VALIDATION_ROWS,
    'epochs': EPOCHS,
    'seed': TRAINING_SEED,
}


In [ ]:
import hashlib, json
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
hub_api = HfApi()
dataset_metadata = hub_api.dataset_info(
    DATASET_REPO_ID, revision=DATASET_REVISION, token=HF_TOKEN
)
dataset_revision = dataset_metadata.sha
assert dataset_revision == DATASET_REVISION
assert getattr(dataset_metadata, 'private', False) is True
manifest_path = Path(hf_hub_download(
    DATASET_REPO_ID,
    filename=f'{DATASET_CONFIG}/manifest.json',
    repo_type='dataset',
    revision=dataset_revision,
    token=HF_TOKEN,
))
manifest_bytes = manifest_path.read_bytes()
manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()
assert manifest_sha256 == MANIFEST_SHA256
manifest = json.loads(manifest_bytes)
assert manifest['dataset_config'] == DATASET_CONFIG
assert manifest['state'] == 'interview_ready'
assert manifest['schema'] == 'distillation-example-v1'
assert manifest['counts'] == EXPECTED_COUNTS
assert manifest['selection']['seed'] == 'babel-interview-2016-v1'
assert manifest['selection']['ordered_identity_sha256'] == EXPECTED_ORDERED_SHA256
assert len(manifest['selection']['smoke_article_keys']) == SMOKE_ROWS
assert manifest['frontier']['complete_corpus'] is False
shard_sha256 = {item['split']: item['sha256'] for item in manifest['shards']}
assert shard_sha256 == EXPECTED_PARQUET_SHA256
assert manifest['selection']['ordered_identity_sha256']['train'] == TRAIN_ORDERED_SHA256
assert manifest['selection']['ordered_identity_sha256']['validation'] == VALIDATION_ORDERED_SHA256
assert manifest['selection']['ordered_identity_sha256']['test'] == TEST_ORDERED_SHA256
readiness_path = Path(hf_hub_download(
    DATASET_REPO_ID,
    filename=f'{DATASET_CONFIG}/readiness.json',
    repo_type='dataset',
    revision=dataset_revision,
    token=HF_TOKEN,
))
readiness_bytes = readiness_path.read_bytes()
dataset_readiness_sha256 = hashlib.sha256(readiness_bytes).hexdigest()
readiness = json.loads(readiness_bytes)
assert readiness['manifest_sha256'] == MANIFEST_SHA256
assert readiness['counts'] == EXPECTED_COUNTS
print('Immutable dataset identity gates PASS; test identity checked as metadata only.')


In [ ]:
from itertools import islice
from datasets import load_dataset
from babel_training.data import validate_training_row

class OrderedInterviewStream:
    def __init__(self, split, row_limit):
        assert split in {'train', 'validation'}
        assert isinstance(row_limit, int) and row_limit > 0
        self.split = split
        self.row_limit = row_limit
        self.next_ordered_row = 0

    def state_dict(self):
        return {
            'state_version': 1,
            'dataset_repo_id': DATASET_REPO_ID,
            'dataset_config': DATASET_CONFIG,
            'dataset_revision': dataset_revision,
            'split': self.split,
            'row_limit': self.row_limit,
            'next_ordered_row': self.next_ordered_row,
        }

    def load_state_dict(self, state):
        expected = {
            'state_version': 1,
            'dataset_repo_id': DATASET_REPO_ID,
            'dataset_config': DATASET_CONFIG,
            'dataset_revision': dataset_revision,
            'split': self.split,
            'row_limit': self.row_limit,
        }
        assert {name: state[name] for name in expected} == expected
        next_ordered_row = state['next_ordered_row']
        assert isinstance(next_ordered_row, int)
        assert 0 <= next_ordered_row <= self.row_limit
        self.next_ordered_row = next_ordered_row

    def __iter__(self):
        start = self.next_ordered_row
        ordered_dataset = load_dataset(
            DATASET_REPO_ID,
            DATASET_CONFIG,
            split=self.split,
            revision=dataset_revision,
            token=HF_TOKEN,
            streaming=True,
        )
        remaining = self.row_limit - start
        for offset, raw in enumerate(islice(ordered_dataset.skip(start), remaining), start=1):
            checked = validate_training_row(raw, expected_split=self.split)
            self.next_ordered_row = start + offset
            yield checked
        assert self.next_ordered_row == self.row_limit, (
            f'{self.split} ended at {self.next_ordered_row}, expected {self.row_limit}'
        )

print('Ordered restartable train/validation stream factory ready; no test stream exists.')


In [ ]:
train_preview = list(islice(OrderedInterviewStream('train', TRAIN_ROWS), 2))
validation_preview = list(islice(
    OrderedInterviewStream('validation', VALIDATION_ROWS), 2
))
assert [row['article_key'] for row in train_preview] == [
    item['article_key']
    for item in manifest['selection']['ordered_identities']['train'][:2]
]
assert [row['article_key'] for row in validation_preview] == [
    item['article_key']
    for item in manifest['selection']['ordered_identities']['validation'][:2]
]
print({
    'train': [
        {'article_key': row['article_key'], 'title': row['canonical_title']}
        for row in train_preview
    ],
    'validation': [
        {'article_key': row['article_key'], 'title': row['canonical_title']}
        for row in validation_preview
    ],
})


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
major, minor = torch.cuda.get_device_capability()
mixed_precision = 'bf16' if major >= 8 else 'fp16'
if major < 8:
    mixed_precision = 'fp16'
print(torch.cuda.get_device_name(0), '| mixed precision:', mixed_precision)


In [ ]:
import math
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoTokenizer
from babel_training.collator import DistillationCollator
from babel_training.model import DistilledQwenEncoder
from babel_training.trainer import DistillationTrainer, build_stateful_train_loader

tokenizer = AutoTokenizer.from_pretrained(
    config.model_id, revision=config.model_revision, token=HF_TOKEN
)
collator = DistillationCollator(tokenizer, max_length=config.max_length)
gate_rows = list(islice(OrderedInterviewStream('validation', VALIDATION_ROWS), 2))
gate_loader = DataLoader(
    gate_rows,
    batch_size=per_device_batch_size,
    collate_fn=collator,
    num_workers=0,
)
validation_batch = next(iter(gate_loader))
smoke_train_stream = OrderedInterviewStream('train', SMOKE_ROWS)
smoke_train_loader = build_stateful_train_loader(
    smoke_train_stream,
    batch_size=per_device_batch_size,
    collate_fn=collator,
)
smoke_model = DistilledQwenEncoder.from_pretrained(config)
smoke_optimizer = AdamW(
    [parameter for parameter in smoke_model.parameters() if parameter.requires_grad],
    lr=2e-4,
)
smoke_scheduler = torch.optim.lr_scheduler.LambdaLR(
    smoke_optimizer, lr_lambda=lambda _: 1.0
)
smoke_trainer = DistillationTrainer(
    smoke_model,
    smoke_train_loader,
    validation_batch=validation_batch,
    model_id=config.model_id,
    model_revision=config.model_revision,
    dataset_revision=dataset_revision,
    training_config=training_config,
    optimizer=smoke_optimizer,
    scheduler=smoke_scheduler,
    mixed_precision=mixed_precision,
    gradient_accumulation_steps=gradient_accumulation_steps,
    max_runtime_minutes=max_runtime_minutes,
)
smoke_scaler = smoke_trainer.accelerator.scaler


In [ ]:
gate = smoke_trainer.one_batch_gate()
assert all(math.isfinite(value) for value in gate.values())
assert gate['gradient_norm'] > 0
print('ONE-BATCH NUMERICAL/GRADIENT GATE PASS', gate)


In [ ]:
smoke_ordered_prefix = list(islice(
    manifest['selection']['ordered_identities']['train'], SMOKE_ROWS
))
assert [item['article_key'] for item in smoke_ordered_prefix] == (
    manifest['selection']['smoke_article_keys']
)
smoke_optimizer_steps = math.ceil(
    SMOKE_ROWS / (per_device_batch_size * gradient_accumulation_steps)
)
smoke_losses = smoke_trainer.train(max_steps=smoke_optimizer_steps)
assert smoke_trainer.global_step == smoke_optimizer_steps
assert smoke_train_stream.next_ordered_row == SMOKE_ROWS
smoke_trainer.epoch = 1
smoke_checkpoint_dir = os.path.join(run_root, 'smoke-checkpoint')
smoke_manifest = smoke_trainer.save(
    smoke_checkpoint_dir,
    metrics={'mode': 'smoke', 'rows': SMOKE_ROWS, 'final_loss': smoke_losses[-1]},
)
torch.save(
    {
        'optimizer': smoke_trainer.optimizer.state_dict(),
        'scheduler': smoke_trainer.scheduler.state_dict(),
        'scaler': smoke_scaler.state_dict() if smoke_scaler is not None else None,
        'rng': {
            'python': __import__('random').getstate(),
            'numpy': __import__('numpy').random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all(),
        },
        'epoch': smoke_trainer.epoch,
        'global_step': smoke_trainer.global_step,
        'next_ordered_row': smoke_train_stream.next_ordered_row,
    },
    os.path.join(smoke_checkpoint_dir, 'notebook_restart_state.pt'),
)
print('SMOKE CHECKPOINT saved after first ordered 1,000 train rows:', smoke_checkpoint_dir)


In [ ]:
import gc, random
import numpy as np

del smoke_trainer, smoke_model, smoke_optimizer, smoke_scheduler, smoke_scaler
del smoke_train_loader, smoke_train_stream
gc.collect()
torch.cuda.empty_cache()
random.seed(TRAINING_SEED)
np.random.seed(TRAINING_SEED)
torch.manual_seed(TRAINING_SEED)
torch.cuda.manual_seed_all(TRAINING_SEED)
production_train_stream = OrderedInterviewStream('train', TRAIN_ROWS)
assert production_train_stream.next_ordered_row == 0
production_train_loader = build_stateful_train_loader(
    production_train_stream,
    batch_size=per_device_batch_size,
    collate_fn=collator,
)
production_model = DistilledQwenEncoder.from_pretrained(config)
production_optimizer = AdamW(
    [
        parameter
        for parameter in production_model.parameters()
        if parameter.requires_grad
    ],
    lr=2e-4,
)
production_scheduler = torch.optim.lr_scheduler.LambdaLR(
    production_optimizer, lr_lambda=lambda _: 1.0
)
production_trainer = DistillationTrainer(
    production_model,
    production_train_loader,
    validation_batch=validation_batch,
    model_id=config.model_id,
    model_revision=config.model_revision,
    dataset_revision=dataset_revision,
    training_config=training_config,
    optimizer=production_optimizer,
    scheduler=production_scheduler,
    mixed_precision=mixed_precision,
    gradient_accumulation_steps=gradient_accumulation_steps,
    max_runtime_minutes=max_runtime_minutes,
)
production_scaler = production_trainer.accelerator.scaler
assert production_trainer.global_step == 0
assert production_trainer.epoch == 0
print('Production model, optimizer, scheduler, scaler, loader cursor, and RNG rebuilt.')


In [ ]:
QUICK_TEST_MODE = True
assert isinstance(QUICK_TEST_MODE, bool)
if QUICK_TEST_MODE is False:
    print('PRODUCTION OPT-IN CONFIRMED: exactly one ordered 50,000-row epoch will run.')
else:
    print('QUICK TEST MODE: exactly one optimizer step will run, save, and stop.')

def capture_restart_state(trainer, stream, scaler):
    return {
        'optimizer': trainer.optimizer.state_dict(),
        'scheduler': trainer.scheduler.state_dict(),
        'scaler': scaler.state_dict() if scaler is not None else None,
        'rng': {
            'python': random.getstate(),
            'numpy': np.random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all(),
        },
        'epoch': trainer.epoch,
        'global_step': trainer.global_step,
        'next_ordered_row': stream.next_ordered_row,
    }

def complete_restartable_checkpoint(path, trainer, stream, scaler, manifest_value):
    restart_state = capture_restart_state(trainer, stream, scaler)
    assert restart_state['next_ordered_row'] == stream.state_dict()['next_ordered_row']
    torch.save(restart_state, os.path.join(path, 'notebook_restart_state.pt'))
    Path(path, 'NOTEBOOK_CHECKPOINT_COMPLETE').write_text(
        'optimizer scheduler scaler rng epoch global_step next_ordered_row\n',
        encoding='utf-8',
    )
    return manifest_value


In [ ]:
for training_pass in range(1):
    if QUICK_TEST_MODE:
        optimizer_step_limit = 1
        quick_start_step = production_trainer.global_step
        quick_losses = production_trainer.train(
            max_steps=quick_start_step + optimizer_step_limit
        )
        assert production_trainer.global_step == quick_start_step + 1
        quick_test_checkpoint_dir = os.path.join(run_root, 'quick-test')
        quick_metrics = {
            'mode': 'quick_test',
            'optimizer_steps': optimizer_step_limit,
            'next_ordered_row': production_train_stream.next_ordered_row,
            'loss': quick_losses[-1],
        }
        quick_manifest = production_trainer.save(
            quick_test_checkpoint_dir,
            metrics=quick_metrics,
        )
        quick_manifest = complete_restartable_checkpoint(
            quick_test_checkpoint_dir, production_trainer,
            production_train_stream, production_scaler, quick_manifest,
        )
        quick_restart_state = torch.load(
            os.path.join(quick_test_checkpoint_dir, 'notebook_restart_state.pt'),
            map_location='cpu',
            weights_only=False,
        )
        assert set(quick_restart_state) == {
            'optimizer', 'scheduler', 'scaler', 'rng', 'epoch',
            'global_step', 'next_ordered_row',
        }
        assert quick_manifest.global_step == 1
        print('================================================================')
        print('QUICK TEST ONLY — FULL EPOCH NOT COMPLETED')
        print('Restartable quick-test checkpoint:', quick_test_checkpoint_dir)
        print('================================================================')
        break

    if not QUICK_TEST_MODE:
        production_optimizer_steps = math.ceil(
            TRAIN_ROWS
            / (per_device_batch_size * gradient_accumulation_steps)
        )
        while production_trainer.global_step < production_optimizer_steps:
            step_target = min(
                production_trainer.global_step + checkpoint_interval,
                production_optimizer_steps,
            )
            chunk_losses = production_trainer.train(max_steps=step_target)
            next_ordered_row = production_train_stream.next_ordered_row
            periodic_checkpoint_dir = os.path.join(
                run_root,
                'production-checkpoints',
                f'step-{production_trainer.global_step:08d}',
            )
            os.makedirs(os.path.dirname(periodic_checkpoint_dir), exist_ok=True)
            periodic_metrics = {
                'mode': 'production_in_progress',
                'latest_loss': chunk_losses[-1],
                'next_ordered_row': next_ordered_row,
            }
            periodic_manifest = production_trainer.save(
                periodic_checkpoint_dir,
                metrics=periodic_metrics,
            )
            complete_restartable_checkpoint(
                periodic_checkpoint_dir, production_trainer,
                production_train_stream, production_scaler, periodic_manifest,
            )
            print(
                'Drive checkpoint',
                production_trainer.global_step,
                '| next ordered row',
                next_ordered_row,
            )
        next_ordered_row = production_train_stream.next_ordered_row
        assert next_ordered_row == TRAIN_ROWS
        assert production_trainer.global_step == production_optimizer_steps
        production_trainer.epoch = EPOCHS
        assert production_trainer.epoch == 1
        print('PRODUCTION: ordered 50,000-row epoch complete')


In [ ]:
if QUICK_TEST_MODE:
    print('Validation skipped: QUICK TEST ONLY — FULL EPOCH NOT COMPLETED')
else:
    from babel_training.validation import validate_embeddings
    fixed_validation_stream = OrderedInterviewStream(
        'validation', VALIDATION_ROWS
    )
    fixed_validation_rows = list(fixed_validation_stream)
    assert len(fixed_validation_rows) == VALIDATION_ROWS
    validation_loader = DataLoader(
        fixed_validation_rows,
        batch_size=per_device_batch_size,
        collate_fn=collator,
        num_workers=0,
    )
    article_keys, student_chunks, teacher_chunks = [], [], []
    production_trainer.model.eval()
    device = next(production_trainer.model.parameters()).device
    with torch.no_grad():
        for batch in validation_loader:
            student_chunks.append(
                production_trainer.model(
                    input_ids=batch['input_ids'].to(device),
                    attention_mask=batch['attention_mask'].to(device),
                ).float().cpu().numpy()
            )
            teacher_chunks.append(batch['teacher_vector'].float().numpy())
            article_keys.extend(batch['article_key'])
    student_vectors = np.concatenate(student_chunks)
    teacher_vectors = np.concatenate(teacher_chunks)
    student_norms = np.linalg.norm(student_vectors, axis=1)
    teacher_norms = np.linalg.norm(teacher_vectors, axis=1)
    invalid_student_vector_count = int(np.count_nonzero(
        ~np.all(np.isfinite(student_vectors), axis=1)
        | ~np.isfinite(student_norms)
        | (student_norms <= 0)
    ))
    invalid_teacher_vector_count = int(np.count_nonzero(
        ~np.all(np.isfinite(teacher_vectors), axis=1)
        | ~np.isfinite(teacher_norms)
        | (teacher_norms <= 0)
    ))
    report = validate_embeddings(
        article_keys,
        student_vectors,
        teacher_vectors,
        dataset_revision=dataset_revision,
        model_revision=config.model_revision,
        tokenizer_revision=config.model_revision,
        dataset_repo_id=DATASET_REPO_ID,
        dataset_config=DATASET_CONFIG,
        dataset_manifest_sha256=MANIFEST_SHA256,
        dataset_readiness_sha256=dataset_readiness_sha256,
        subset='fixed-interview-5000',
    )
    invalid_vector_count = report.invalid_vector_count
    validation_summary = {
        **report.to_dict(),
        'invalid_student_vector_count': invalid_student_vector_count,
        'invalid_teacher_vector_count': invalid_teacher_vector_count,
        'invalid_vector_count': invalid_vector_count,
    }
    validation_report_path = os.path.join(run_root, 'validation-report.json')
    Path(validation_report_path).write_text(
        json.dumps(validation_summary, sort_keys=True, separators=(',', ':')) + '\n',
        encoding='utf-8',
    )
    print('VALIDATION PASS: 5,000 fixed rows', validation_summary['metrics'])
    print('Invalid vectors:', {
        'student': invalid_student_vector_count,
        'teacher': invalid_teacher_vector_count,
        'joint': invalid_vector_count,
    })


In [ ]:
if QUICK_TEST_MODE:
    print('Production-final save skipped: quick-test checkpoint is already restartable.')
else:
    def checkpoint_tree_sha256(path):
        digest = hashlib.sha256()
        root = Path(path)
        for item in sorted(candidate for candidate in root.rglob('*') if candidate.is_file()):
            digest.update(item.relative_to(root).as_posix().encode('utf-8'))
            digest.update(b'\0')
            digest.update(hashlib.sha256(item.read_bytes()).digest())
        return digest.hexdigest()

    final_checkpoint_dir = os.path.join(run_root, 'production-final')
    final_checkpoint_fingerprint = production_trainer.validation_fingerprint()
    final_metrics = {
        **report.metrics,
        'invalid_student_vector_count': invalid_student_vector_count,
        'invalid_teacher_vector_count': invalid_teacher_vector_count,
        'invalid_vector_count': invalid_vector_count,
        'next_ordered_row': production_train_stream.next_ordered_row,
    }
    final_manifest = production_trainer.save(
        final_checkpoint_dir,
        metrics=final_metrics,
    )
    final_manifest = complete_restartable_checkpoint(
        final_checkpoint_dir, production_trainer, production_train_stream,
        production_scaler, final_manifest,
    )
    assert final_manifest.epoch == EPOCHS
    assert final_manifest.loader_state
    final_checkpoint_identity = checkpoint_tree_sha256(final_checkpoint_dir)
    print('Saved production-final restartable checkpoint:', final_checkpoint_identity)


In [ ]:
if QUICK_TEST_MODE:
    print('Final reload skipped in quick-test mode.')
else:
    fingerprint_before_reload = final_checkpoint_fingerprint
    production_trainer.reload(final_checkpoint_dir)
    fingerprint_after_reload = production_trainer.validation_fingerprint()
    assert fingerprint_after_reload == fingerprint_before_reload
    assert production_trainer.global_step == production_optimizer_steps
    assert production_train_stream.next_ordered_row == TRAIN_ROWS
    print('Final checkpoint reload/fingerprint PASS')


In [ ]:
if QUICK_TEST_MODE:
    print('Isolated resume verification is reserved for the production run.')
else:
    import shutil
    periodic_checkpoints = sorted(
        Path(run_root, 'production-checkpoints').glob('step-*')
    )
    resumable_source = next(
        path
        for path in reversed(periodic_checkpoints)
        if int(path.name.removeprefix('step-')) < production_optimizer_steps
        and json.loads(Path(path, 'manifest.json').read_text())['loader_state']
    )
    resume_verification_dir = os.path.join(run_root, 'resume-verification')
    shutil.copytree(resumable_source, resume_verification_dir)
    final_identity_before_resume = checkpoint_tree_sha256(final_checkpoint_dir)
    copied_identity_before_resume = checkpoint_tree_sha256(resume_verification_dir)
    production_trainer.reload(resume_verification_dir)
    saved_step = production_trainer.global_step
    resume_losses = production_trainer.train(max_steps=saved_step + 1)
    assert production_trainer.global_step == saved_step + 1
    assert resume_losses
    assert checkpoint_tree_sha256(resume_verification_dir) == copied_identity_before_resume
    assert checkpoint_tree_sha256(final_checkpoint_dir) == final_identity_before_resume
    production_trainer.reload(final_checkpoint_dir)
    assert production_trainer.validation_fingerprint() == final_checkpoint_fingerprint
    print('Isolated one-step resume PASS; production-final checkpoint remained immutable.')


In [ ]:
if QUICK_TEST_MODE:
    print('Artifact export skipped: quick-test weights are not production artifacts.')
else:
    from safetensors.torch import save_file

    def canonical_json_bytes(value):
        return (
            json.dumps(
                value,
                sort_keys=True,
                separators=(',', ':'),
                ensure_ascii=False,
                allow_nan=False,
            ) + '\n'
        ).encode('utf-8')

    final_student = production_trainer.accelerator.unwrap_model(
        production_trainer.model
    )
    projection_tensors, adapter_tensors, adapter_config = (
        final_student.export_components()
    )
    assert projection_tensors['weight'].shape == (100, 1024)
    assert projection_tensors['bias'].shape == (100,)
    assert adapter_tensors
    artifact_dir = Path(run_root, 'distilled-artifact')
    artifact_dir.mkdir()
    save_file(projection_tensors, artifact_dir / 'projection.safetensors')
    save_file(adapter_tensors, artifact_dir / 'adapter_model.safetensors')
    complete_training_config = {
        **training_config,
        'quick_test_mode': False,
        'completed_ordered_train_rows': TRAIN_ROWS,
        'completed_epochs': EPOCHS,
        'final_global_step': production_trainer.global_step,
    }
    export_documents = {
        'adapter_config.json': adapter_config,
        'training_config.json': complete_training_config,
        'validation_report.json': validation_summary,
        'final_checkpoint_identity.json': {
            'schema_version': 1,
            'tree_sha256': final_checkpoint_identity,
            'epoch': final_manifest.epoch,
            'global_step': final_manifest.global_step,
            'next_ordered_row': production_train_stream.next_ordered_row,
        },
    }
    for filename, document in export_documents.items():
        Path(artifact_dir, filename).write_bytes(canonical_json_bytes(document))
    artifact_hashes = {
        path.name: hashlib.sha256(path.read_bytes()).hexdigest()
        for path in sorted(artifact_dir.iterdir())
        if path.is_file()
    }
    artifact_identity = {
        'artifact_schema': 'babel-distillation-2016-interview-v1',
        'source': {'commit_sha': SOURCE_COMMIT_SHA},
        'model': {
            'id': config.model_id,
            'revision': MODEL_REVISION,
            'tokenizer_revision': MODEL_REVISION,
        },
        'dataset': {
            'repo_id': DATASET_REPO_ID,
            'config': 'distillation_2016_interview',
            'commit_sha': DATASET_REVISION,
            'manifest_sha256': MANIFEST_SHA256,
            'readiness_sha256': dataset_readiness_sha256,
            'counts': EXPECTED_COUNTS,
            'ordered_identity_sha256': EXPECTED_ORDERED_SHA256,
            'parquet_sha256': EXPECTED_PARQUET_SHA256,
            'test_usage': 'identity metadata only; examples unopened',
        },
        'protocol': {
            'smoke_rows': SMOKE_ROWS,
            'train_rows': TRAIN_ROWS,
            'validation_rows': VALIDATION_ROWS,
            'epochs': EPOCHS,
            'max_length': config.max_length,
        },
        'lora': adapter_config,
        'projection': {'input_dimension': 1024, 'output_dimension': 100},
        'training_config': complete_training_config,
        'validation': validation_summary,
        'final_checkpoint': export_documents['final_checkpoint_identity.json'],
        'artifact_hashes': artifact_hashes,
    }
    artifact_id = hashlib.sha256(canonical_json_bytes(artifact_identity)).hexdigest()
    artifact_manifest = {
        'artifact_id': artifact_id,
        'immutable': True,
        **artifact_identity,
        'publication': {
            'repo_id': DESTINATION_MODEL_REPO,
            'private': True,
            'artifact_payload_commit_sha': None,
        },
    }
    artifact_manifest_path = artifact_dir / 'artifact_manifest.json'
    artifact_manifest_path.write_bytes(canonical_json_bytes(artifact_manifest))
    print('Immutable LoRA + 100d projection export prepared:', artifact_id)
    print('Artifact hashes:', artifact_hashes)


In [ ]:
if QUICK_TEST_MODE:
    print('Hub publication skipped: quick-test artifacts are never published.')
else:
    import re
    from huggingface_hub import CommitOperationAdd

    publish_api = HfApi()
    publish_api.create_repo(
        repo_id=DESTINATION_MODEL_REPO,
        repo_type='model',
        private=True,
        exist_ok=True,
        token=HF_TOKEN,
    )
    private_repo_info = publish_api.model_info(
        DESTINATION_MODEL_REPO, revision='main', token=HF_TOKEN
    )
    assert getattr(private_repo_info, 'private', False) is True
    artifact_prefix = f'artifacts/{artifact_id}/'
    existing_paths = {
        sibling.rfilename for sibling in getattr(private_repo_info, 'siblings', [])
    }
    assert not any(path.startswith(artifact_prefix) for path in existing_paths), (
        'Immutable artifact path already exists; refusing to overwrite it.'
    )
    payload_paths = sorted(
        path for path in artifact_dir.iterdir()
        if path.is_file() and path.name != 'artifact_manifest.json'
    )
    payload_commit = publish_api.create_commit(
        repo_id=DESTINATION_MODEL_REPO,
        repo_type='model',
        revision='main',
        parent_commit=private_repo_info.sha,
        operations=[
            CommitOperationAdd(
                path_in_repo=artifact_prefix + path.name,
                path_or_fileobj=str(path),
            )
            for path in payload_paths
        ],
        commit_message=f'Publish interview artifact payload {artifact_id}',
        token=HF_TOKEN,
    )
    artifact_payload_commit_sha = payload_commit.oid
    assert re.fullmatch(r'[a-f0-9]{40}', artifact_payload_commit_sha)
    artifact_manifest['publication']['artifact_payload_commit_sha'] = (
        artifact_payload_commit_sha
    )
    artifact_manifest_path.write_bytes(canonical_json_bytes(artifact_manifest))
    artifact_manifest_sha256 = hashlib.sha256(
        artifact_manifest_path.read_bytes()
    ).hexdigest()
    manifest_commit = publish_api.create_commit(
        repo_id=DESTINATION_MODEL_REPO,
        repo_type='model',
        revision='main',
        parent_commit=artifact_payload_commit_sha,
        operations=[
            CommitOperationAdd(
                path_in_repo=artifact_prefix + 'artifact_manifest.json',
                path_or_fileobj=str(artifact_manifest_path),
            )
        ],
        commit_message=f'Finalize interview artifact manifest {artifact_id}',
        token=HF_TOKEN,
    )
    hub_commit_sha = manifest_commit.oid
    assert re.fullmatch(r'[a-f0-9]{40}', hub_commit_sha)
    verified_private_info = publish_api.model_info(
        DESTINATION_MODEL_REPO, revision=hub_commit_sha, token=HF_TOKEN
    )
    assert getattr(verified_private_info, 'private', False) is True
    assert verified_private_info.sha == hub_commit_sha
    print('PRIVATE HUB PUBLICATION PASS')
    print('Repository:', DESTINATION_MODEL_REPO)
    print('Artifact payload commit SHA:', artifact_payload_commit_sha)
    print('Final manifest commit SHA:', hub_commit_sha)
    print('Final artifact manifest SHA-256:', artifact_manifest_sha256)
